# Расчет количества ПА в соответствии с методикой проекта 700 приказа

In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd

In [ ]:
G = ox.load_graphml(r'C:\QGIS\Novosibirsk\highwys_nsk_40126516_cfa4_4993_89f4_dd4940fb30c6.ml')
print(G.number_of_nodes(), G.number_of_edges())

114482 242831


In [ ]:

## Загрузка пожарных подразделений
units = gpd.read_file(r'C:\QGIS\Novosibirsk\data\units.gpkg', layer='__add_10000')
print(len(units))
units = ox.projection.project_gdf(units)
### Сопоставление пожарных подразделений с ближайшими узлами ГДС
units['node'] = ox.distance.nearest_nodes(ox.project_graph(G), units.geometry.x, units.geometry.y)
### Формирование словаря подразделений для передачи алгоритму
units_dict = {k:v for k,v in zip(units['node'], units['name'])}

10


In [ ]:
## Загрузка зданий
buildings = gpd.read_file(r'C:\QGIS\Novosibirsk\data\buildings_nsk.gpkg')
print(len(buildings))
### Сопоставление зданий с ближайшими узлами ГДС
### Для полигональных объектов следует использовать центроиды: `geometry.centroid`
### Наборы данных желательно перепроецировать в локальную СК
centroids = ox.projection.project_gdf(buildings).geometry.centroid
buildings['node'] = ox.distance.nearest_nodes(ox.project_graph(G), 
                                              centroids.x,
                                              centroids.y
                                              )

76578


In [ ]:
living_classes = [
    'Жилые дома',
    'Многоквартирные жилые дома'
]
living_bld = buildings.query('building_ru in @living_classes').copy()
print('Всего' , len(buildings), 'из них жилых', len(living_bld))

Всего 76578 из них жилых 32820


In [ ]:
living_bld

,BUILDING,ADDR_CITY,A_STRT,A_SBRB,A_HSNMBR,A_PLACE,A_PSTCD,B_LEVELS,NAME,NAME_EN,NAME_RU,OSM_TYPE,OSM_ID,levels,building_ru,geometry,node,pop
2,apartments,None,улица Героев Труда,None,35А,None,630055,None,None,None,None,relation,11124099,1,Многоквартирные жилые дома,"MULTIPOLYGON (((83.07885 54.8693, 83.07867 54....",6344,269.992530
8,apartments,Новосибирск,Кавалерийская улица,None,25,None,None,None,None,None,None,relation,12791009,1,Многоквартирные жилые дома,"MULTIPOLYGON (((82.92898 55.05849, 82.92985 55...",71237,288.538634
10,apartments,None,улица Дунаевского,None,3,None,630027,10,None,None,None,relation,13292500,10,Многоквартирные жилые дома,"MULTIPOLYGON (((82.97282 55.10069, 82.97295 55...",21412,322.566383
11,apartments,None,Нарымская улица,None,21,None,None,12,None,None,None,relation,11134199,12,Многоквартирные жилые дома,"MULTIPOLYGON (((82.90625 55.04268, 82.90607 55...",6412,327.572553
12,apartments,None,улица Владимира Заровного,None,26,None,630083,10,None,None,None,relation,13923551,10,Многоквартирные жилые дома,"MULTIPOLYGON (((82.96931 54.99084, 82.96896 54...",35391,300.188712
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76438,apartments,Новосибирск,Волочаевская улица,None,53,None,630010,10,None,None,None,relation,6398337,10,Многоквартирные жилые дома,"MULTIPOLYGON (((83.00366 55.05118, 83.00351 55...",43527,373.808246
76449,apartments,Новосибирск,Заречная улица,None,4,None,630030,10,None,None,None,relation,3920012,10,Многоквартирные жилые дома,"MULTIPOLYGON (((83.04584 54.97752, 83.04579 54...",83251,1212.026915
76452,apartments,Новосибирск,улица Титова,None,240/1,None,None,10,None,None,None,relation,3928999,10,Многоквартирные жилые дома,"MULTIPOLYGON (((82.80817 54.97993, 82.808 54.9...",24830,363.427120
76454,apartments,Новосибирск,улица Маяковского,None,1/1,None,630037,5,None,None,None,relation,5837072,5,Многоквартирные жилые дома,"MULTIPOLYGON (((83.10226 54.97043, 83.10242 54...",13486,415.042734


In [ ]:
# Определяем численность населения в жилых домах
population = 1_500_000

total_area = sum(living_bld.area * living_bld['levels'])
pop_per_m = population / total_area
print(total_area, pop_per_m)

living_bld['pop'] = living_bld.area * pop_per_m
living_bld['pop'].describe()

C:\Users\Admin\AppData\Local\Temp\ipykernel_10036\2487150741.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  total_area = sum(living_bld.area * living_bld['levels'])


0.0042047250205056175 356741521.1898031


C:\Users\Admin\AppData\Local\Temp\ipykernel_10036\2487150741.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  living_bld['pop'] = living_bld.area * pop_per_m


count    32820.000000
mean        10.849032
std         18.189471
min          0.710176
25%          4.323291
50%          5.399484
75%          7.697205
max        472.812357
Name: pop, dtype: float64

## Расчеты

### Расчет прикрытия населения

In [ ]:
from genesis.states import FirstArrivalUnitState

times, nearest = FirstArrivalUnitState()(env=G, points=units_dict)

living_bld = living_bld.merge(nearest,
                              left_on='node',
                              right_index=True)
living_bld.head(2)

,BUILDING,ADDR_CITY,A_STRT,A_SBRB,A_HSNMBR,A_PLACE,A_PSTCD,B_LEVELS,NAME,NAME_EN,NAME_RU,OSM_TYPE,OSM_ID,levels,building_ru,geometry,node,pop,nearest
2,apartments,None,улица Героев Труда,None,35А,None,630055,None,None,None,None,relation,11124099,1,Многоквартирные жилые дома,"MULTIPOLYGON (((83.07885 54.8693, 83.07867 54....",6344,64.089966,3
8,apartments,Новосибирск,Кавалерийская улица,None,25,None,None,None,None,None,None,relation,12791009,1,Многоквартирные жилые дома,"MULTIPOLYGON (((82.92898 55.05849, 82.92985 55...",71237,68.492381,1


In [ ]:
living_bld.groupby('nearest')['pop'].sum()

nearest
1     60533.621812
10    17684.617597
2     39298.316035
3     42647.564963
4     31471.856913
5     27300.467209
6     24329.663034
7     13252.894541
8     43436.094103
9     31391.994547
Name: pop, dtype: float64

In [ ]:
units = units.merge(living_bld.groupby('nearest')['pop'].sum(),
            left_on='name',
            right_index=True)
units

,osmid,x,y,name,geometry,node,pop
0,3399,624097.960367,6.101464e+06,1,POINT (624097.96 6101463.949),20281,60533.621812
1,9559,621094.194579,6.094806e+06,2,POINT (621094.195 6094806.317),73570,39298.316035
2,5386,630978.016348,6.079893e+06,3,POINT (630978.016 6079893.351),38552,42647.564963
3,9361,630240.269308,6.093945e+06,4,POINT (630240.269 6093945.348),72620,31471.856913
4,1283,624135.720677,6.089784e+06,5,POINT (624135.721 6089783.929),4052,27300.467209
5,7616,617143.263084,6.095906e+06,6,POINT (617143.263 6095906.275),58889,24329.663034
6,8032,635614.331143,6.091602e+06,7,POINT (635614.331 6091602.418),61543,13252.894541
7,6773,627034.382985,6.099874e+06,8,POINT (627034.383 6099873.931),48998,43436.094103
8,4117,620967.570210,6.106896e+06,9,POINT (620967.57 6106896.311),23334,31391.994547
9,12054,626124.502464,6.108747e+06,10,POINT (626124.502 6108746.859),91772,17684.617597


### Определение требуемого расхода

In [ ]:
def get_need_Q(pop):
    if pop <= 1_000:
        return 15
    if pop <= 5_000:
        return 20
    if pop <= 10_000:
        return 40
    if pop <= 25_000:
        return 50
    if pop <= 50_000:
        return 60
    if pop <= 100_000:
        return 70
    if pop <= 200_000:
        return 120
    
    return None

get_need_Q(60533.621812)

70

In [ ]:
units['Q'] = units['pop'].apply(get_need_Q)
units

,osmid,x,y,name,geometry,node,pop,Q
0,3399,624097.960367,6.101464e+06,1,POINT (624097.96 6101463.949),20281,60533.621812,70
1,9559,621094.194579,6.094806e+06,2,POINT (621094.195 6094806.317),73570,39298.316035,60
2,5386,630978.016348,6.079893e+06,3,POINT (630978.016 6079893.351),38552,42647.564963,60
3,9361,630240.269308,6.093945e+06,4,POINT (630240.269 6093945.348),72620,31471.856913,60
4,1283,624135.720677,6.089784e+06,5,POINT (624135.721 6089783.929),4052,27300.467209,60
5,7616,617143.263084,6.095906e+06,6,POINT (617143.263 6095906.275),58889,24329.663034,50
6,8032,635614.331143,6.091602e+06,7,POINT (635614.331 6091602.418),61543,13252.894541,50
7,6773,627034.382985,6.099874e+06,8,POINT (627034.383 6099873.931),48998,43436.094103,60
8,4117,620967.570210,6.106896e+06,9,POINT (620967.57 6106896.311),23334,31391.994547,60
9,12054,626124.502464,6.108747e+06,10,POINT (626124.502 6108746.859),91772,17684.617597,50


### Расчет коэффициента К4

где k4 – коэффициент, учитывающий наличие в районе выезда территориального подразделения пожарной охраны организаций, технологические процессы которых связаны с обращением взрывопожароопасных, пожароопасных веществ и материалов, безр. Определяется по табл. 1.2.2;

### Расчет К5

k5 – коэффициент, учитывающий наличие в районе выезда территориального подразделения пожарной охраны безводных участков, безр. Определяется по табл. 1.2.3;

### Расчет К6

k6 – коэффициент, учитывающий наличие в районе выезда территориального подразделения пожарной охраны зданий и сооружений V степени огнестойкости, горение которых может привести к распространению пожара на соседние здания и сооружения, безр. Определяется по табл. 1.2.4.

